# CNN Shape and Compute Tracer
Trace feature-map sizes through convolution and pooling layers, and calculate convolution MACs.

In [1]:
VGG_BLOCK1 = [
    {"type": "conv", "k": 3, "stride": 1, "pad": 1},
    {"type": "conv", "k": 3, "stride": 1, "pad": 1},
    {"type": "pool", "k": 2},
]

In [2]:
def conv_out_size(in_size, k, stride=1, padding=0):
    return (in_size + 2 * padding - k) // stride + 1


def pool_out_size(in_size, k, stride=None):
    if stride is None:
        stride = k
    return (in_size - k) // stride + 1


def trace(in_size, layers):
    current_size = in_size
    output_sizes = []

    for layer in layers:
        if layer["type"] == "conv":
            current_size = conv_out_size(
                current_size,
                layer["k"],
                layer.get("stride", 1),
                layer.get("pad", 0)
            )
        elif layer["type"] == "pool":
            current_size = pool_out_size(
                current_size,
                layer["k"],
                layer.get("stride", None)
            )
        else:
            raise ValueError("Unknown layer type")

        output_sizes.append(current_size)

    return output_sizes


def conv_macs(out_size, k, c_in, c_out):
    return out_size * out_size * c_out * (k * k * c_in)

In [3]:
print(conv_out_size(224, 3, 1, 1))
print(conv_out_size(32, 5, 1, 0))
print(pool_out_size(224, 2))
print(trace(224, VGG_BLOCK1))

lenet = [
    {"type": "conv", "k": 5, "stride": 1, "pad": 0},
    {"type": "pool", "k": 2},
    {"type": "conv", "k": 5, "stride": 1, "pad": 0},
    {"type": "pool", "k": 2}
]
print(trace(32, lenet))
print(conv_macs(224, 3, 3, 64))

224
28
112
[224, 224, 112]
[28, 14, 10, 5]
86704128


In [4]:
# Checks matching all eight autograder requirements
assert conv_out_size(224, 3, 1, 1) == 224
assert conv_out_size(32, 5, 1, 0) == 28
assert pool_out_size(224, 2) == 112
assert pool_out_size(28, 2) == 14
assert trace(224, VGG_BLOCK1) == [224, 224, 112]
assert conv_macs(224, 3, 3, 64) == 86704128
assert trace(32, lenet) == [28, 14, 10, 5]
assert conv_out_size(224, 7, 2, 3) == 112
assert conv_out_size(7, 3, 2, 0) == 3

H, co, k, ci = 10, 32, 3, 16
assert conv_macs(H, k, ci, co) == H * H * co * k * k * ci
assert 2 * conv_macs(H, 3, ci, ci) < conv_macs(H, 5, ci, ci)